# A09 — Descongelamento Progressivo do ResNet50

**Objetivo:** Comparar estrategias de descongelamento progressivo (unfreeze) do backbone ResNet50 para classificacao binaria ASTER.

| Estrategia | Camadas treinaveis | LR backbone |
|---|---|---|
| **E1: Congelado** | Apenas adapter + head | 1e-3 |
| **E2: Unfreeze layer4** | adapter + head + layer4 | 1e-4 |
| **E3: Unfreeze layer3+4** | adapter + head + layer3 + layer4 | 1e-4 |

## 1. Setup

In [ ]:
!pip install -q --upgrade torch torchvision scikit-learn pandas matplotlib

## 2. Dados

Upload `pixels_dataset.csv` e `extracted_codes.json` pelo painel lateral do Colab.

In [ ]:
import os

CSV_PATH = '/content/pixels_dataset.csv'
CODES_PATH = '/content/extracted_codes.json'

assert os.path.isfile(CSV_PATH), f'Nao encontrou {CSV_PATH}'
assert os.path.isfile(CODES_PATH), f'Nao encontrou {CODES_PATH}'
print('Dados encontrados!')

In [ ]:
import json
import re
import copy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.v2 as T
import torchvision.models as models
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    f1_score,
)
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 3. Pipeline de Dados

In [ ]:
def get_ordered_pixel_columns(df, pixel_prefix='pixel_'):
    pixel_cols = [c for c in df.columns if c.startswith(pixel_prefix)]
    if not pixel_cols:
        raise ValueError(f'Nenhuma coluna com prefixo {pixel_prefix}')
    def _sort_key(col):
        suffix = col[len(pixel_prefix):]
        return (0, int(suffix)) if suffix.isdigit() else (1, suffix)
    pixel_cols.sort(key=_sort_key)
    return pixel_cols


def infer_cnn_shape(df, pixel_columns=None, n_channels=None, height=None, width=None):
    pixel_columns = pixel_columns or get_ordered_pixel_columns(df)
    n_pixels_total = len(pixel_columns)
    if n_channels is None and 'count' in df.columns:
        u = df['count'].dropna().unique()
        if len(u) == 1 and int(u[0]) > 0:
            n_channels = int(u[0])
    if height is None and 'height' in df.columns:
        u = df['height'].dropna().unique()
        if len(u) == 1 and int(u[0]) > 0:
            height = int(u[0])
    if width is None and 'width' in df.columns:
        u = df['width'].dropna().unique()
        if len(u) == 1 and int(u[0]) > 0:
            width = int(u[0])
    if n_channels and height and width:
        assert n_channels * height * width == n_pixels_total
        return {'n_channels': n_channels, 'height': height, 'width': width, 'pixels_per_channel': height * width}
    if n_channels and not height:
        ppc = n_pixels_total // n_channels
        side = int(np.sqrt(ppc))
        assert side * side == ppc
        return {'n_channels': n_channels, 'height': side, 'width': side, 'pixels_per_channel': ppc}
    raise ValueError('Nao foi possivel inferir shape.')


def dataframe_to_cnn_tensor(df, pixel_prefix='pixel_', n_channels=None, height=None, width=None):
    pixel_columns = get_ordered_pixel_columns(df, pixel_prefix)
    shape_info = infer_cnn_shape(df, pixel_columns, n_channels, height, width)
    flat = df[pixel_columns].to_numpy(dtype=np.float32, copy=True)
    n_samples = flat.shape[0]
    x = flat.reshape(n_samples, shape_info['n_channels'], shape_info['height'], shape_info['width'])
    x = np.transpose(x, (0, 2, 3, 1))
    return x, shape_info


def labels_from_extracted_codes(paths, codes_path):
    with open(codes_path, 'r') as f:
        codes = json.load(f)
    positives = set(codes.get('positivos', []))
    negatives = set(codes.get('negativos', []))
    all_ids = sorted(positives | negatives, key=len, reverse=True)
    pattern = '|'.join(re.escape(c) for c in all_ids)
    path_series = pd.Series(list(paths), dtype=str)
    image_ids = path_series.str.extract(rf'({pattern})', expand=False)
    labels = np.where(image_ids.isin(positives), 1, np.where(image_ids.isin(negatives), 0, -1)).astype(np.int64)
    image_ids = image_ids.fillna('').to_numpy(dtype=object)
    return labels, image_ids


def fit_channel_normalizer(x, eps=1e-8):
    axes = (0, 1, 2)
    mean = x.mean(axis=axes, keepdims=True).astype(np.float32)
    std = x.std(axis=axes, keepdims=True).astype(np.float32)
    std = np.where(std < eps, 1.0, std)
    return {'mean': mean, 'std': std}


def apply_channel_normalizer(x, normalizer):
    return ((x - normalizer['mean']) / normalizer['std']).astype(np.float32)


def stratified_group_split(image_ids, labels, test_size=0.15, val_size=0.15, seed=42):
    frame = pd.DataFrame({'image_id': np.asarray(image_ids, dtype=str), 'label': np.asarray(labels, dtype=np.int64)})
    frame = frame[frame['image_id'] != ''].copy()
    img_level = frame.groupby('image_id')['label'].agg(lambda v: int(pd.Series(v).mode().iloc[0])).reset_index()
    train_val_ids, test_ids = train_test_split(img_level['image_id'], test_size=test_size, random_state=seed, stratify=img_level['label'])
    tv_frame = img_level[img_level['image_id'].isin(train_val_ids)]
    train_ids, val_ids = train_test_split(tv_frame['image_id'], test_size=val_size/(1-test_size), random_state=seed, stratify=tv_frame['label'])
    ids_arr = np.asarray(image_ids, dtype=object)
    return {
        'train_idx': np.flatnonzero(np.isin(ids_arr, train_ids.to_numpy())),
        'val_idx': np.flatnonzero(np.isin(ids_arr, val_ids.to_numpy())),
        'test_idx': np.flatnonzero(np.isin(ids_arr, test_ids.to_numpy())),
    }


def prepare_data(csv_path, codes_path, test_size=0.15, val_size=0.15, seed=42):
    print('Carregando CSV...')
    df = pd.read_csv(csv_path)
    print(f'  {df.shape[0]} amostras, {df.shape[1]} colunas')
    X, shape_info = dataframe_to_cnn_tensor(df)
    print(f'  Tensor: {X.shape} (N, H, W, C)')
    labels, image_ids = labels_from_extracted_codes(df['path'], codes_path)
    valid = (labels != -1) & (np.asarray(image_ids, dtype=str) != '')
    X, labels, image_ids = X[valid], labels[valid], image_ids[valid]
    print(f'  Validos: {X.shape[0]}')
    split = stratified_group_split(image_ids, labels, test_size, val_size, seed)
    ti, vi, tei = split['train_idx'], split['val_idx'], split['test_idx']
    result = {
        'X_train': X[ti], 'y_train': labels[ti],
        'X_val': X[vi], 'y_val': labels[vi],
        'X_test': X[tei], 'y_test': labels[tei],
        'shape_info': shape_info,
    }
    for name in ['train', 'val', 'test']:
        y = result[f'y_{name}']
        print(f'  {name:5s}: {result[f"X_{name}"].shape} | classes: {dict(zip(*np.unique(y, return_counts=True)))}')
    return result

print('Funcoes carregadas!')

In [ ]:
data = prepare_data(CSV_PATH, CODES_PATH)
X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']

normalizer = fit_channel_normalizer(X_train)
X_train = apply_channel_normalizer(X_train, normalizer)
X_val = apply_channel_normalizer(X_val, normalizer)
X_test = apply_channel_normalizer(X_test, normalizer)
print('Normalizacao z-score aplicada')

In [ ]:
class ASTERDataset(Dataset):
    def __init__(self, X, y, augment=False):
        self.X = torch.from_numpy(X).permute(0, 3, 1, 2).float()
        self.y = torch.from_numpy(y).float()
        self.augment = augment
        if augment:
            self.transform = nn.Sequential(
                T.RandomHorizontalFlip(p=0.5),
                T.RandomVerticalFlip(p=0.5),
                T.RandomAffine(degrees=15, translate=(0.05, 0.05)),
            )

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx]
        if self.augment:
            x = self.transform(x)
        return x, self.y[idx]


BATCH_SIZE = 16
train_ds = ASTERDataset(X_train, y_train, augment=True)
val_ds = ASTERDataset(X_val, y_val, augment=False)
test_ds = ASTERDataset(X_test, y_test, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
print(f'DataLoaders prontos | Batch: {BATCH_SIZE}')

## 4. Modelo e Funcoes de Treino

In [ ]:
class ResNetClassifier(nn.Module):
    """
    Transfer Learning: ASTER 9 bandas -> ResNet50 -> Classificacao Binaria.
    Suporta descongelamento seletivo de camadas do backbone.
    """

    def __init__(self, dropout_rate=0.3):
        super().__init__()

        self.channel_adapter = nn.Sequential(
            nn.Conv2d(9, 3, kernel_size=1, bias=False),
            nn.BatchNorm2d(3),
            nn.ReLU(inplace=True),
        )

        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        # Separar camadas nomeadas para controle granular
        self.layer0 = nn.Sequential(backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool)
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4
        self.avgpool = backbone.avgpool

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout_rate),
            nn.Linear(2048, 1),
        )

    def freeze_all(self):
        """Congela todo o backbone."""
        for layer in [self.layer0, self.layer1, self.layer2, self.layer3, self.layer4]:
            for param in layer.parameters():
                param.requires_grad = False

    def unfreeze_layers(self, layer_names):
        """Descongela camadas especificas do backbone."""
        layer_map = {
            'layer0': self.layer0,
            'layer1': self.layer1,
            'layer2': self.layer2,
            'layer3': self.layer3,
            'layer4': self.layer4,
        }
        for name in layer_names:
            for param in layer_map[name].parameters():
                param.requires_grad = True

    def count_params(self):
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return total, trainable

    def forward(self, x):
        x = self.channel_adapter(x)
        x = F.interpolate(x, size=(224, 224), mode='bilinear', align_corners=False)
        x = self.layer0(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = self.head(x)
        return x.squeeze(-1)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        logits = model(X_b)
        loss = criterion(logits, y_b)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y_b)
        correct += ((torch.sigmoid(logits) > 0.5).float() == y_b).sum().item()
        total += len(y_b)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []
    for X_b, y_b in loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        logits = model(X_b)
        loss = criterion(logits, y_b)
        total_loss += loss.item() * len(y_b)
        probs = torch.sigmoid(logits)
        correct += ((probs > 0.5).float() == y_b).sum().item()
        total += len(y_b)
        all_probs.extend(probs.cpu().numpy())
        all_labels.extend(y_b.cpu().numpy())
    probs_arr, labels_arr = np.array(all_probs), np.array(all_labels)
    metrics = {'loss': total_loss/total, 'accuracy': correct/total, 'probs': probs_arr, 'labels': labels_arr}
    try:
        metrics['auc_roc'] = roc_auc_score(labels_arr, probs_arr)
        metrics['f1'] = f1_score(labels_arr, (probs_arr > 0.5).astype(int), average='weighted')
    except ValueError:
        metrics['auc_roc'] = 0.0
        metrics['f1'] = 0.0
    return metrics


def run_experiment(model, train_loader, val_loader, device, lr=1e-3, epochs=20, patience=5, name='exp'):
    """Treina modelo e retorna historico + metricas."""
    total, trainable = model.count_params()
    print(f'\n{"=" * 60}')
    print(f'EXPERIMENTO: {name}')
    print(f'Params: {total:,} total | {trainable:,} treinaveis ({100*trainable/total:.1f}%)')
    print(f'LR: {lr} | Epochs: {epochs} | Patience: {patience}')
    print(f'{"=" * 60}')

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    best_val_loss = float('inf')
    patience_counter = 0
    best_state = None
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_auc': []}

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val = evaluate(model, val_loader, criterion, device)

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val['loss'])
        history['val_acc'].append(val['accuracy'])
        history['val_auc'].append(val['auc_roc'])

        print(f"  Epoch {epoch+1:02d}/{epochs} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.3f} | "
              f"Val Loss: {val['loss']:.4f} Acc: {val['accuracy']:.3f} AUC: {val['auc_roc']:.3f}")

        if val['loss'] < best_val_loss:
            best_val_loss = val['loss']
            patience_counter = 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'  Early stopping na epoch {epoch+1}')
                break

    model.load_state_dict(best_state)
    return history

print('Funcoes de treino carregadas!')

## 5. Experimento E1 — Backbone Congelado (Baseline)

In [ ]:
model_e1 = ResNetClassifier(dropout_rate=0.3)
model_e1.freeze_all()
model_e1 = model_e1.to(device)

history_e1 = run_experiment(
    model_e1, train_loader, val_loader, device,
    lr=1e-3, epochs=20, patience=5, name='E1: Congelado (baseline)'
)

## 6. Experimento E2 — Unfreeze layer4

In [ ]:
model_e2 = ResNetClassifier(dropout_rate=0.3)
model_e2.freeze_all()
model_e2.unfreeze_layers(['layer4'])
model_e2 = model_e2.to(device)

history_e2 = run_experiment(
    model_e2, train_loader, val_loader, device,
    lr=1e-4, epochs=20, patience=5, name='E2: Unfreeze layer4'
)

## 7. Experimento E3 — Unfreeze layer3 + layer4

In [ ]:
model_e3 = ResNetClassifier(dropout_rate=0.3)
model_e3.freeze_all()
model_e3.unfreeze_layers(['layer3', 'layer4'])
model_e3 = model_e3.to(device)

history_e3 = run_experiment(
    model_e3, train_loader, val_loader, device,
    lr=1e-4, epochs=20, patience=5, name='E3: Unfreeze layer3+4'
)

## 8. Avaliacao no Teste — Todos os Modelos

In [ ]:
criterion = nn.BCEWithLogitsLoss()
results = {}

for name, mdl in [('E1: Congelado', model_e1), ('E2: layer4', model_e2), ('E3: layer3+4', model_e3)]:
    m = evaluate(mdl, test_loader, criterion, device)
    results[name] = m
    print(f"\n{name}")
    print(f"  Accuracy: {m['accuracy']:.4f} | F1: {m['f1']:.4f} | AUC-ROC: {m['auc_roc']:.4f}")
    y_pred = (m['probs'] > 0.5).astype(int)
    print(f"  Confusion Matrix:\n{confusion_matrix(m['labels'], y_pred)}")

In [ ]:
# Tabela comparativa
print('\n' + '=' * 60)
print('COMPARACAO FINAL — CONJUNTO DE TESTE')
print('=' * 60)
print(f'{"Estrategia":<20} {"Accuracy":>10} {"F1-Score":>10} {"AUC-ROC":>10}')
print('-' * 52)
for name, m in results.items():
    print(f'{name:<20} {m["accuracy"]:>10.4f} {m["f1"]:>10.4f} {m["auc_roc"]:>10.4f}')

## 9. Graficos Comparativos

In [ ]:
histories = {
    'E1: Congelado': history_e1,
    'E2: layer4': history_e2,
    'E3: layer3+4': history_e3,
}
colors = {'E1: Congelado': '#1f77b4', 'E2: layer4': '#ff7f0e', 'E3: layer3+4': '#2ca02c'}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for name, h in histories.items():
    axes[0].plot(h['val_loss'], label=name, color=colors[name])
axes[0].set_title('Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for name, h in histories.items():
    axes[1].plot(h['val_acc'], label=name, color=colors[name])
axes[1].set_title('Val Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

for name, h in histories.items():
    axes[2].plot(h['val_auc'], label=name, color=colors[name])
axes[2].set_title('Val AUC-ROC')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

fig.suptitle('Descongelamento Progressivo — Comparacao de Estrategias', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Grafico de barras — metricas finais no teste
names = list(results.keys())
acc = [results[n]['accuracy'] for n in names]
f1s = [results[n]['f1'] for n in names]
aucs = [results[n]['auc_roc'] for n in names]

x = np.arange(len(names))
width = 0.25

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width, acc, width, label='Accuracy', color='#1f77b4')
ax.bar(x, f1s, width, label='F1-Score', color='#ff7f0e')
ax.bar(x + width, aucs, width, label='AUC-ROC', color='#2ca02c')

ax.set_ylabel('Score')
ax.set_title('Metricas no Teste — Descongelamento Progressivo')
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.legend()
ax.set_ylim(0.5, 1.0)
ax.grid(True, alpha=0.3, axis='y')

for i, (a, f, u) in enumerate(zip(acc, f1s, aucs)):
    ax.text(i - width, a + 0.01, f'{a:.3f}', ha='center', fontsize=8)
    ax.text(i, f + 0.01, f'{f:.3f}', ha='center', fontsize=8)
    ax.text(i + width, u + 0.01, f'{u:.3f}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()